## ZCB Bond Yield Curve

### Miniproject 2 | Valuation for Financial Engineering.

Team:
- Jiayi Chen
- Esteban López Araiza Bravo

### Assumptions  
We will make the rates continuous for easy calculations.




In [1]:
# Libraries
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path
from datetime import datetime

# Change working directory to the project root
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
    
PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))

print("PROJECT ROOT:", PROJECT_ROOT)

PROJECT ROOT: c:\Users\samla\zcb_treasury_curve


Next, check the raw data

In [2]:
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'

bond_data = pd.read_excel(RAW_ROOT / 'Treasury curve 090426.xlsx')

bond_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 353 entries, 0 to 352
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Maturity     353 non-null    datetime64[ns]
 1   Coupon       353 non-null    float64       
 2   Bid          353 non-null    float64       
 3   Asked        353 non-null    float64       
 4   Chg          353 non-null    object        
 5   Asked yield  353 non-null    float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 16.7+ KB


In [3]:
bond_data.head(10)

,Maturity,Coupon,Bid,Asked,Chg,Asked yield
0,2026-09-15,4.625,100.000,100.010,-0.002,2.937
1,2026-09-30,0.875,99.256,99.266,0.01,3.625
2,2026-09-30,1.625,99.276,99.286,0.004,3.318
3,2026-09-30,3.500,99.306,99.316,0.002,3.603
4,2026-10-15,4.625,100.022,100.032,-0.002,3.576
5,2026-10-31,1.125,99.186,99.196,0.01,3.808
6,2026-10-31,1.625,99.212,99.222,0.008,3.755
7,2026-10-31,4.125,100.000,100.010,-0.006,3.877
8,2026-11-15,2.000,99.204,99.214,0.004,3.787
9,2026-11-15,4.625,100.040,100.050,-0.002,3.741


As we can see, the ``Chg`` column should be float64 too, this means there are some values that are not numbers, must be ``NULL``.

In [4]:
bad = bond_data[~bond_data['Chg'].map(pd.api.types.is_float)]
print(bad)

      Maturity  Coupon      Bid    Asked    Chg  Asked yield
18  2027-01-15   4.000  100.010  100.020  unch.        3.807
26  2027-02-28   4.125  100.016  100.026  unch.        3.940
31  2027-04-15   4.500  100.082  100.092  unch.        4.002
33  2027-04-30   2.750   99.044   99.054  unch.        4.061
42  2027-06-30   3.250   99.092   99.102  unch.        4.107
59  2027-10-31   0.500   95.260   95.270  unch.        4.260
263 2040-08-15   1.125   60.274   60.294  unch.        5.069
273 2041-11-15   3.125   79.044   79.064  unch.        5.110
280 2042-08-15   3.375   80.224   80.244  unch.        5.160
281 2042-11-15   2.750   73.144   73.164  unch.        5.188
286 2043-05-15   3.875   85.116   85.136  unch.        5.190
330 2051-02-15   1.875   52.294   52.314  unch.        5.343
336 2052-08-15   3.000   67.190   67.210  unch.        5.311
340 2053-08-15   4.125   83.120   83.140  unch.        5.285
347 2055-05-15   4.750   92.136   92.156  unch.        5.259
352 2056-08-15   5.125  

The non float values seem to be "unch" wich means unchanged, so they can be changed to 0.

In [13]:
bond_data.loc[:, 'Chg'] = bond_data.loc[:, 'Chg'].replace('unch.', 0.0).astype(float)

## Dirty prices

### Assumptions

- We also know that the asked price is the clean price, but for this project we need dirty prices, this means clean prices + accrued interest. We will first get the accrued interest and then create a new column for the Dirty Price.   
- We also know that the bonds and notes in the excel file pay coupons every six months.  
- As the data is of 09/04/2026, the curve, the accrued days and the dirty price should all be at the same date
- The decimals for bid and Asked price are in Treasury form (the first two should be divided by 32 and after that divided by 256). See [IRS's Valuation of Government Securities](https://www.irs.gov/pub/irs-tege/teb2a03.pdf) and [FAQs about Security BuyBacks](https://treasurydirect.gov/help-center/faqs/buyback-faqs/)
- We will use the ``asked`` column as the clean price before changing to decimal instead of Treasury form
- The principal is 100

In [6]:
# import utils
from src.utils import treasury_form_2_decimal

valuation_date = datetime(2026, 9, 4)  # September 4, 2026

bond_data['Clean Price'] = bond_data['Asked'].apply(treasury_form_2_decimal)
bond_data.head(10)

,Maturity,Coupon,Bid,Asked,Chg,Asked yield,Clean Price
0,2026-09-15,4.625,100.000,100.010,-0.002,2.937,100.031250
1,2026-09-30,0.875,99.256,99.266,0.01,3.625,99.835938
2,2026-09-30,1.625,99.276,99.286,0.004,3.318,99.898438
3,2026-09-30,3.500,99.306,99.316,0.002,3.603,99.992188
4,2026-10-15,4.625,100.022,100.032,-0.002,3.576,100.097656
5,2026-10-31,1.125,99.186,99.196,0.01,3.808,99.613281
6,2026-10-31,1.625,99.212,99.222,0.008,3.755,99.691406
7,2026-10-31,4.125,100.000,100.010,-0.006,3.877,100.031250
8,2026-11-15,2.000,99.204,99.214,0.004,3.787,99.667969
9,2026-11-15,4.625,100.040,100.050,-0.002,3.741,100.160156


In [7]:
# sanity check 
clean_price1 = bond_data.loc[0, 'Clean Price']
manual1 = 100 + 1 / 32 + 0 / 256

clean_price2 = bond_data.loc[1, 'Clean Price']
manual2 = 99 + 26 / 32 + 6 / 256
print('Sanity Check:')
print(f"Function Clean Price 1: {clean_price1}, Manual Clean Price 1: {manual1}")
print(f"Function Clean Price 2: {clean_price2}, Manual Clean Price 2: {manual2}")

Sanity Check:
Function Clean Price 1: 100.03125, Manual Clean Price 1: 100.03125
Function Clean Price 2: 99.8359375, Manual Clean Price 2: 99.8359375


In [8]:
# Get Days to Maturity
bond_data['DTM'] = (pd.to_datetime(bond_data['Maturity'], format='%m/%d/%y') - valuation_date).dt.days
bond_data.head(10)

,Maturity,Coupon,Bid,Asked,Chg,Asked yield,Clean Price,DTM
0,2026-09-15,4.625,100.000,100.010,-0.002,2.937,100.031250,11
1,2026-09-30,0.875,99.256,99.266,0.01,3.625,99.835938,26
2,2026-09-30,1.625,99.276,99.286,0.004,3.318,99.898438,26
3,2026-09-30,3.500,99.306,99.316,0.002,3.603,99.992188,26
4,2026-10-15,4.625,100.022,100.032,-0.002,3.576,100.097656,41
5,2026-10-31,1.125,99.186,99.196,0.01,3.808,99.613281,57
6,2026-10-31,1.625,99.212,99.222,0.008,3.755,99.691406,57
7,2026-10-31,4.125,100.000,100.010,-0.006,3.877,100.031250,57
8,2026-11-15,2.000,99.204,99.214,0.004,3.787,99.667969,72
9,2026-11-15,4.625,100.040,100.050,-0.002,3.741,100.160156,72


AS we can see, we have multiple bonds with the same maturity, there are two ways we can decide which ones to use. We could use only the ones which asked yield is closer to the coupon rate for normal bootstrapping or we can use all bonds with the Svensson model. 

### Accrual days and interest
As treasury notes and bonds use Actual/Actual for cashflows, this means each accrual day should reflect the actual days, i.e. 180, 181, 182.  
For the accrual interest we will first get the weighted accrual days for the actual number of days in the coupon an then multiply it by the coupon rate divided by two, considering its a semiannual rate.


In [9]:
from src.bond_funs import current_coupon_dates
bond_data[["Accrual Start", "Accrual End"]] = bond_data.apply(
    lambda row: current_coupon_dates(
        row["Maturity"],
        valuation_date,
    ),
    axis=1,
)
bond_data["Accrued Days"] = (
    valuation_date - bond_data["Accrual Start"]
).dt.days

bond_data["Coupon Period Days"] = (
    bond_data["Accrual End"] - bond_data["Accrual Start"]
).dt.days

bond_data["Accrued Fraction"] = (
    bond_data["Accrued Days"] / bond_data["Coupon Period Days"]
)
bond_data["Accrued Interest"] = (
    bond_data["Coupon"] / 2
    * bond_data["Accrued Fraction"]
)

bond_data[["Accrual Start", "Accrual End", "Accrued Days", "Coupon Period Days", "Accrued Fraction", "Accrued Interest"]].head(10)

,Accrual Start,Accrual End,Accrued Days,Coupon Period Days,Accrued Fraction,Accrued Interest
0,2026-03-15,2026-09-15,173,184,0.940217,2.174253
1,2026-03-31,2026-09-30,157,183,0.857923,0.375342
2,2026-03-31,2026-09-30,157,183,0.857923,0.697063
3,2026-03-31,2026-09-30,157,183,0.857923,1.501366
4,2026-04-15,2026-10-15,142,183,0.775956,1.794399
5,2026-04-30,2026-10-31,127,184,0.690217,0.388247
6,2026-04-30,2026-10-31,127,184,0.690217,0.560802
7,2026-04-30,2026-10-31,127,184,0.690217,1.423573
8,2026-05-15,2026-11-15,112,184,0.608696,0.608696
9,2026-05-15,2026-11-15,112,184,0.608696,1.407609


### Dirty Price

Clean price + accrued interest


In [10]:
bond_data['Dirty Price'] = bond_data['Clean Price'] + bond_data['Accrued Interest']

bond_data[['Clean Price', 'Accrued Interest', 'Dirty Price']].head(10)

,Clean Price,Accrued Interest,Dirty Price
0,100.031250,2.174253,102.205503
1,99.835938,0.375342,100.211279
2,99.898438,0.697063,100.595500
3,99.992188,1.501366,101.493554
4,100.097656,1.794399,101.892055
5,99.613281,0.388247,100.001529
6,99.691406,0.560802,100.252208
7,100.031250,1.423573,101.454823
8,99.667969,0.608696,100.276664
9,100.160156,1.407609,101.567765


In [11]:
# Save data
bond_data.to_csv(PROJECT_ROOT / 'data' / 'processed' / 'bond_data.csv', index=False)    

## Curve building model
For this project we decided to use the Svensson model as it is the one that most central banks use. This is the model we will use:

$$DF(t) = e^{-z(t)t}$$

where:
- $z(t)$ is the spot zero continuous rate
- $DF(t)$ is the discount factor
- $t$ is the time in years since the valuation date

Then, the theoretical dirty price for any given bond is:  
$$
Dirty^{model}_i = \sum^{N_i}_{j=1}{CF_{ij}e^{-z(t_{ij})t_{ij}}}
$$
Where the last cashflow includes the principal and the coupon
$$ CF_{i,N} = 100 + \frac{Coupon_i}{2} $$

and:
- $i$ is the bond index, with $i \in \{ 1, 2, \ldots, N \}$
- $CF_{i,j}$ is the cash flow $j$ for the bond $i$
- $N_i$ is the number of the remaining cashflows for bond $i$
- $Coupon_i$ is the annual coupon amount per $100 of principal.

The formula for $z(t_{ij})$ using Svensson model is:

$$
\begin{aligned}
z(t_{i,j})
={}&
\beta_0
+
\beta_1
\left(
\frac{1-e^{-t_{i,j}/\tau_1}}
{t_{i,j}/\tau_1}
\right)
\\[4pt]
&+
\beta_2
\left(
\frac{1-e^{-t_{i,j}/\tau_1}}
{t_{i,j}/\tau_1}
-
e^{-t_{i,j}/\tau_1}
\right)
\\[4pt]
&+
\beta_3
\left(
\frac{1-e^{-t_{i,j}/\tau_2}}
{t_{i,j}/\tau_2}
-
e^{-t_{i,j}/\tau_2}
\right)
\end{aligned}
$$

The model has six parameters:

$$
\theta=
(\beta_0,\beta_1,\beta_2,\beta_3,\tau_1,\tau_2)
$$

where:
- $\beta_0$ is the long-term rate
- $\beta_1$ is the short-term slope 
- $\beta_2$ is the first curvature
- $\beta_3$ is thesecond curvature
- $\tau_1$ determines when will $\beta_2$ will appear in years
- $\tau_2$ determines when will $\beta_3$ will appear in years

These parameters are common to all bonds and are estimated by minimizing the difference between observed and model-implied dirty prices:
$$
\widehat{\theta}
=
\underset{\theta}{\operatorname{arg\,min}}
\sum_{i=1}^{N}
\left(
Dirty_i^{observed}
-
Dirty_i^{model}(\theta)
\right)^2
$$

### Next Steps
- Make Dirty Price Function
- Make Svensson zero rate function
- Make residual function
- Sanity check for a single bond
- Perform minimizer using least squares 